# MEDUSA Reproduction Colab notebook

This notebook serves as a thin launcher. All logic is contained in the `.py` files inside the `code/` directory.

In [1]:
from google.colab import drive
import sys

path = "/content/gdrive/MyDrive/CS4782/final_proj" # change based on where files are

drive.mount('/content/gdrive')

base_dir = path
sys.path.append(base_dir)
%cd $path

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content/gdrive/MyDrive/CS4782/final_proj


In [ ]:

# Environment setup — run ONCE on a fresh runtime, then RESTART the session.

# Step 1 — pinned tokenizer + protobuf stack
!pip install -q --force-reinstall --no-deps \
    'transformers==4.46.3' 'tokenizers==0.20.3' 'sentencepiece==0.2.0' \
    'protobuf>=3.20.3,<5.0'

# Step 2 — other requirements (leaves Colab torch alone)
!pip install -q --upgrade-strategy only-if-needed -r requirements.txt

# Step 3 — FastChat for conversation templates
!pip install -q --no-deps fschat

# Sanity check + restart prompt
import transformers, tokenizers, sentencepiece, google.protobuf
from fastchat.conversation import get_conv_template
print(f'transformers={transformers.__version__}  '
      f'tokenizers={tokenizers.__version__}  '
      f'sentencepiece={sentencepiece.__version__}  '
      f'protobuf={google.protobuf.__version__}')
print('FastChat Vicuna template loaded:', get_conv_template('vicuna_v1.1').name)


In [ ]:
# Full training run (paper-scale). Use --max_samples 1000 for a quick smoke test.
# H100 speedup knobs: batch_size * grad_accum_steps held at 32 (paper-equivalent
# effective batch). SDPA + frozen-backbone no_grad make batch=16 trivially fit.
%run code/train.py --max_samples 60000 --save_path medusa_heads_tinyllama.pt --batch_size 16 --grad_accum_steps 2 --compile

Using device: cuda
Loading base model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


torch.compile enabled.
Trainable parameters: 278,921,216
Preparing data...
Loading dataset...


Generating train split:   0%|          | 0/120675 [00:00<?, ? examples/s]

Preprocessing with chat template (model=TinyLlama/TinyLlama-1.1B-Chat-v1.0)...


tokenize:   0%|          | 0/60000 [00:00<?, ?it/s]

Kept 60000 samples (skipped 0 empty/invalid).
Train samples: 54000, Val samples: 6000
Loss weights per head: [0.8, 0.6400000000000001, 0.5120000000000001, 0.4096000000000001]


/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


Epoch 1 | Step 50/3375 | Loss: 23.0553 | Head Acc: ['0.038', '0.025', '0.027', '0.025']
Epoch 1 | Step 100/3375 | Loss: 15.6736 | Head Acc: ['0.103', '0.051', '0.042', '0.036']
Epoch 1 | Step 150/3375 | Loss: 13.1590 | Head Acc: ['0.155', '0.076', '0.057', '0.048']
Epoch 1 | Step 200/3375 | Loss: 12.2717 | Head Acc: ['0.194', '0.098', '0.071', '0.059']
Epoch 1 | Step 250/3375 | Loss: 11.8266 | Head Acc: ['0.222', '0.114', '0.082', '0.067']
Epoch 1 | Step 300/3375 | Loss: 10.9854 | Head Acc: ['0.244', '0.128', '0.091', '0.074']
Epoch 1 | Step 350/3375 | Loss: 11.1584 | Head Acc: ['0.260', '0.137', '0.097', '0.078']
Epoch 1 | Step 400/3375 | Loss: 10.2327 | Head Acc: ['0.275', '0.146', '0.103', '0.083']
Epoch 1 | Step 450/3375 | Loss: 10.8330 | Head Acc: ['0.286', '0.154', '0.108', '0.087']
Epoch 1 | Step 500/3375 | Loss: 10.7466 | Head Acc: ['0.296', '0.160', '0.112', '0.090']
Epoch 1 | Step 550/3375 | Loss: 11.0709 | Head Acc: ['0.305', '0.166', '0.116', '0.092']
Epoch 1 | Step 600/337

In [3]:
%run code/benchmark.py --mode greedy

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


=== Greedy Baseline ===
  [Write a Python script to sort ...] tps=69.63 time=1.84s tokens=128
  [Explain the theory of relativi...] tps=69.12 time=1.85s tokens=128
  [What are the benefits of eatin...] tps=65.59 time=1.95s tokens=128
  [Compose a short poem about the...] tps=66.20 time=1.93s tokens=128
  [Give me a step-by-step recipe ...] tps=69.07 time=1.85s tokens=128
Average greedy TPS: 67.92


# Medusa Inference (Greedy Acceptance)

Run the full propose → verify → accept loop with greedy acceptance.  
Requires `results/medusa_heads_tinyllama.pt` from the training cell above.  
Prints per-prompt acceptance rate (extra tree tokens accepted per step) and TPS.

In [4]:
%run code/benchmark.py --mode medusa --acceptance greedy --tree_budget 64 --checkpoint results/medusa_heads_tinyllama.pt

Using device: cuda

=== Medusa Inference ===


We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)


Loaded head checkpoint from results/medusa_heads_tinyllama.pt
  [Write a Python script to sort ...] acc=1.91 tps=145.5 | Here's a Python script that sorts a list:  ```python # Example list: [1, 2, 3, 4
  [Explain the theory of relativi...] acc=1.23 tps=122.8 | The theory of relativity is a scientific theory that explains how objects move a
  [What are the benefits of eatin...] acc=1.87 tps=157.7 | Here are some benefits of eating healthy:  1. Improved physical health: Eating a
  [Compose a short poem about the...] acc=1.11 tps=113.8 | The moon, a silent witness to our lives, A celestial sight that never fades, A b
  [Give me a step-by-step recipe ...] acc=1.58 tps=136.7 | Ingredients: - 2 cups all-purpose flour - 1/2 teaspoon baking soda - 1/2 teaspoo

Average Medusa TPS: 135.29
Average extra tokens accepted per step: 1.540
Results saved to /content/gdrive/MyDrive/CS4782/final_proj/results/medusa_inference.json


## Full TinyLlama Benchmark

Runs greedy baseline + Medusa inference + per-head accuracy in one pass.  
Saves `results/TinyLlama-1.1B-Chat-v1.0_benchmark.json` with speedup ratio and all metrics.

In [5]:
%run code/benchmark.py --mode full --model_id TinyLlama/TinyLlama-1.1B-Chat-v1.0 --max_new_tokens 128 --checkpoint results/medusa_heads_tinyllama.pt

Using device: cuda

=== Full Benchmark — TinyLlama/TinyLlama-1.1B-Chat-v1.0 ===
Loaded head checkpoint from results/medusa_heads_tinyllama.pt

--- Greedy baseline ---
  [Write a Python script to sort ...] tps=71.02 time=1.80s tokens=128
  [Explain the theory of relativi...] tps=72.53 time=1.76s tokens=128
  [What are the benefits of eatin...] tps=71.99 time=1.78s tokens=128
  [Compose a short poem about the...] tps=70.20 time=1.82s tokens=128
  [Give me a step-by-step recipe ...] tps=71.53 time=1.79s tokens=128
Avg greedy TPS: 71.45

--- Medusa inference (greedy acceptance, 64-node tree, design=paper) ---
  [Write a Python script to sort ...] acc=1.91 tps=163.1 | Here's a Python script that sorts a list:  ```python # Example list: [1, 2, 3, 4
  [Explain the theory of relativi...] acc=1.23 tps=124.2 | The theory of relativity is a scientific theory that explains how objects move a
  [What are the benefits of eatin...] acc=1.87 tps=161.7 | Here are some benefits of eating healthy:  1. Im

## TinyLlama — Typical Acceptance & Tree Budget Tuning

Compare greedy vs typical acceptance criterion (paper §2.3.1) and tune the tree budget.  
Runs below save to `results/comparison_tinyllama_budget64.json` and `results/comparison_tinyllama_budget32.json`.

In [6]:
# Greedy vs typical acceptance — 64-node tree
%run code/benchmark.py --mode medusa --compare --tree_budget 64 --checkpoint results/medusa_heads_tinyllama.pt
!mv -f results/comparison.json results/comparison_tinyllama_budget64.json

Using device: cuda

=== Medusa Inference ===
Loaded head checkpoint from results/medusa_heads_tinyllama.pt

--- Acceptance: greedy ---
  [Write a Python script to sort ...] acc=1.91 tps=163.4 | Here's a Python script that sorts a list:  ```python # Example list: [1, 2, 3, 4
  [Explain the theory of relativi...] acc=1.23 tps=126.3 | The theory of relativity is a scientific theory that explains how objects move a
  [What are the benefits of eatin...] acc=1.87 tps=159.7 | Here are some benefits of eating healthy:  1. Improved physical health: Eating a
  [Compose a short poem about the...] acc=1.11 tps=112.0 | The moon, a silent witness to our lives, A celestial sight that never fades, A b
  [Give me a step-by-step recipe ...] acc=1.58 tps=140.1 | Ingredients: - 2 cups all-purpose flour - 1/2 teaspoon baking soda - 1/2 teaspoo
  => avg acceptance_rate=1.540  avg_tps=140.29

--- Acceptance: typical ---
  [Write a Python script to sort ...] acc=2.00 tps=151.6 | Here is a Python script that s

In [7]:
# Tree budget tuning — 32-node tree for speed vs acceptance tradeoff
%run code/benchmark.py --mode medusa --compare --tree_budget 32 --checkpoint results/medusa_heads_tinyllama.pt
!mv -f results/comparison.json results/comparison_tinyllama_budget32.json

Using device: cuda

=== Medusa Inference ===
Loaded head checkpoint from results/medusa_heads_tinyllama.pt

--- Acceptance: greedy ---
  [Write a Python script to sort ...] acc=1.67 tps=157.3 | Here's a Python script that sorts a list:  ```python # Example list: [1, 2, 3, 4
  [Explain the theory of relativi...] acc=1.13 tps=129.3 | The theory of relativity is a scientific theory that explains how objects move a
  [What are the benefits of eatin...] acc=1.63 tps=154.6 | Here are some benefits of eating healthy:  1. Improved physical health: Eating a
  [Compose a short poem about the...] acc=1.08 tps=121.1 | The moon, a silent witness to our lives, A celestial sight that never fades, A b
  [Give me a step-by-step recipe ...] acc=1.37 tps=138.6 | Ingredients: - 2 cups all-purpose flour - 1/2 teaspoon baking soda - 1/2 teaspoo
  => avg acceptance_rate=1.377  avg_tps=140.19

--- Acceptance: typical ---
  [Write a Python script to sort ...] acc=1.78 tps=156.9 | Here is a Python script that s

## TinyLlama — Table 3 Ablation (heads-only / naive tree / optimized tree)

Reproduces paper Table 3 rows 1–3: speedup contribution of each technique.
- Row 1 (`tree=none`): heads-only linear chain, no tree attention — paper target ~1.54x.
- Row 2 (`tree=naive`): full 220-node Cartesian product — paper target ~1.92x.
- Row 3 (`tree=optimized`): 64-node pruned tree (MEDUSA-1 headline) — paper target ~2.18x.

In [8]:
# Table 3 ablation — runs {none, naive, optimized} + shared greedy baseline.
# Saves results/table3_TinyLlama-1.1B-Chat-v1.0.json (consumed by visualize.py).
%run code/benchmark.py --mode table3 --model_id TinyLlama/TinyLlama-1.1B-Chat-v1.0 --max_new_tokens 128 --checkpoint results/medusa_heads_tinyllama.pt

Using device: cuda

=== Table 3 Ablation — TinyLlama/TinyLlama-1.1B-Chat-v1.0 ===
Loaded head checkpoint from results/medusa_heads_tinyllama.pt

--- Greedy baseline (shared across ablation rows) ---
  [Write a Python script to sort ...] tps=73.20 time=1.75s tokens=128
  [Explain the theory of relativi...] tps=70.83 time=1.81s tokens=128
  [What are the benefits of eatin...] tps=71.63 time=1.79s tokens=128
  [Compose a short poem about the...] tps=71.60 time=1.79s tokens=128
  [Give me a step-by-step recipe ...] tps=72.85 time=1.76s tokens=128
Avg greedy TPS: 72.02

--- Medusa tree=none ---
  [Write a Python script to sort ...] acc=1.35 tps=146.0 | Here's a Python script that sorts a list:  ```python # Example list: [1, 2, 3, 4
  [Explain the theory of relativi...] acc=0.57 tps=97.9 | The theory of relativity is a scientific theory that describes how the laws of p
  [What are the benefits of eatin...] acc=0.90 tps=120.0 | Here are some benefits of eating healthy:  1. Improved physical h

## Vicuna-7B Scale Run

Paper-scale reproduction target. Backbone is loaded in 4-bit (bitsandbytes nf4) via `--quantize` so the model fits on a 24 GB GPU (L4 / A100 40GB).  
Trains fresh heads on Vicuna-7B and saves them to `results/medusa_heads_vicuna.pt` (kept separate from the TinyLlama checkpoint), then runs the full greedy + Medusa + head-accuracy benchmark.

In [9]:
# Paper-faithful Vicuna-7B training. Dropping --quantize on H100 80GB — the paper
# trained in fp16/bf16, and nf4 dequant is actually SLOWER than bf16 on H100 because
# H100 has no native nf4 tensor cores (every matmul pays a dequant tax). For ≤24GB
# GPUs, re-add --quantize.
# Effective batch held at 32 (8 × 4) to match the TinyLlama/paper setup.
%run code/train.py --model_name lmsys/vicuna-7b-v1.5 --max_samples 60000 --save_path medusa_heads_vicuna.pt --batch_size 8 --grad_accum_steps 4 --compile

Using device: cuda
Loading base model: lmsys/vicuna-7b-v1.5


tokenizer_config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

ShareGPT_V4.3_unfiltered_cleaned_split.j(…):   0%|          | 0.00/464M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120675 [00:00<?, ? examples/s]

Preprocessing with chat template (model=lmsys/vicuna-7b-v1.5)...


tokenize:   0%|          | 0/60000 [00:00<?, ?it/s]

Kept 60000 samples (skipped 0 empty/invalid).
Train samples: 54000, Val samples: 6000
Loss weights per head: [0.8, 0.6400000000000001, 0.5120000000000001, 0.4096000000000001]
Epoch 1 | Step 50/6750 | Loss: 33.5360 | Head Acc: ['0.023', '0.020', '0.022', '0.025']
Epoch 1 | Step 100/6750 | Loss: 21.9466 | Head Acc: ['0.053', '0.027', '0.024', '0.024']
Epoch 1 | Step 150/6750 | Loss: 17.5098 | Head Acc: ['0.099', '0.041', '0.030', '0.028']
Epoch 1 | Step 200/6750 | Loss: 13.4505 | Head Acc: ['0.135', '0.055', '0.038', '0.032']
Epoch 1 | Step 250/6750 | Loss: 12.8377 | Head Acc: ['0.168', '0.071', '0.048', '0.039']
Epoch 1 | Step 300/6750 | Loss: 12.9687 | Head Acc: ['0.195', '0.085', '0.057', '0.045']
Epoch 1 | Step 350/6750 | Loss: 10.5215 | Head Acc: ['0.216', '0.096', '0.063', '0.049']
Epoch 1 | Step 400/6750 | Loss: 11.8201 | Head Acc: ['0.234', '0.106', '0.070', '0.054']
Epoch 1 | Step 450/6750 | Loss: 11.5803 | Head Acc: ['0.250', '0.116', '0.076', '0.058']
Epoch 1 | Step 500/6750 |

In [10]:
%run code/benchmark.py --mode full --model_id lmsys/vicuna-7b-v1.5 --max_new_tokens 128 --checkpoint results/medusa_heads_vicuna.pt

Using device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuratio


=== Full Benchmark — lmsys/vicuna-7b-v1.5 ===
Loaded head checkpoint from results/medusa_heads_vicuna.pt

--- Greedy baseline ---
  [Write a Python script to sort ...] tps=51.99 time=2.46s tokens=128


model.safetensors.index.json: 0.00B [00:00, ?B/s]

  [Explain the theory of relativi...] tps=50.53 time=2.53s tokens=128
  [What are the benefits of eatin...] tps=52.13 time=2.34s tokens=122
  [Compose a short poem about the...] tps=52.63 time=2.43s tokens=128
  [Give me a step-by-step recipe ...] tps=51.21 time=2.50s tokens=128
Avg greedy TPS: 51.70

--- Medusa inference (greedy acceptance, 64-node tree, design=paper) ---
  [Write a Python script to sort ...] acc=1.98 tps=119.3 | Here is a simple Python script to sort a list: ```python def sort_list(my_list):
  [Explain the theory of relativi...] acc=1.31 tps=98.0 | The theory of relativity is a scientific concept that explains how gravity works
  [What are the benefits of eatin...] acc=1.54 tps=102.4 | Eating a healthy diet offers numerous benefits for overall health and wellness, 
  [Compose a short poem about the...] acc=1.00 tps=80.8 | Silent, luminous, Glowing in the dark of night, The moon, a guiding light.  Risi
  [Give me a step-by-step recipe ...] acc=2.12 tps=128.5 | Sure, h

## Vicuna-7B — Typical Acceptance & Tree Budget Tuning

Same greedy-vs-typical + tree-budget sweep as TinyLlama, but on the paper-scale backbone.  
Runs below save to `results/comparison_vicuna_budget64.json` and `results/comparison_vicuna_budget32.json`.

In [11]:
# Greedy vs typical acceptance — 64-node tree
%run code/benchmark.py --mode medusa --compare --tree_budget 64 --model_id lmsys/vicuna-7b-v1.5 --checkpoint results/medusa_heads_vicuna.pt
!mv -f results/comparison.json results/comparison_vicuna_budget64.json

Using device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


=== Medusa Inference ===
Loaded head checkpoint from results/medusa_heads_vicuna.pt

--- Acceptance: greedy ---
  [Write a Python script to sort ...] acc=1.98 tps=122.4 | Here is a simple Python script to sort a list: ```python def sort_list(my_list):
  [Explain the theory of relativi...] acc=1.31 tps=93.7 | The theory of relativity is a scientific concept that explains how gravity works
  [What are the benefits of eatin...] acc=1.54 tps=103.3 | Eating a healthy diet offers numerous benefits for overall health and wellness, 
  [Compose a short poem about the...] acc=1.00 tps=78.4 | Silent, luminous, Glowing in the dark of night, The moon, a guiding light.  Risi
  [Give me a step-by-step recipe ...] acc=2.12 tps=124.3 | Sure, here's a recipe for chocolate chip cookies:  Ingredients:  * 2 1/4 cups al
  => avg acceptance_rate=1.589  avg_tps=104.42

--- Acceptance: typical ---
  [Write a Python script to sort ...] acc=2.88 tps=153.8 | Here is a simple Python script that sorts a list in as

In [12]:
# Tree budget tuning — 32-node tree for speed vs acceptance tradeoff
%run code/benchmark.py --mode medusa --compare --tree_budget 32 --model_id lmsys/vicuna-7b-v1.5 --checkpoint results/medusa_heads_vicuna.pt
!mv -f results/comparison.json results/comparison_vicuna_budget32.json

Using device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


=== Medusa Inference ===
Loaded head checkpoint from results/medusa_heads_vicuna.pt

--- Acceptance: greedy ---
  [Write a Python script to sort ...] acc=1.91 tps=124.0 | Here is a simple Python script to sort a list: ```python def sort_list(my_list):
  [Explain the theory of relativi...] acc=1.27 tps=98.1 | The theory of relativity is a scientific concept that explains how gravity works
  [What are the benefits of eatin...] acc=1.39 tps=102.9 | Eating a healthy diet offers numerous benefits for overall health and wellness, 
  [Compose a short poem about the...] acc=0.98 tps=84.0 | Silent, luminous, Glowing in the dark of night, The moon, a guiding light.  Risi
  [Give me a step-by-step recipe ...] acc=1.89 tps=121.4 | Sure, here's a recipe for chocolate chip cookies:  Ingredients:  * 2 1/4 cups al
  => avg acceptance_rate=1.488  avg_tps=106.07

--- Acceptance: typical ---
  [Write a Python script to sort ...] acc=2.37 tps=141.3 | Here is a simple Python script that sorts a list of in

## Vicuna-7B — Table 3 Ablation

Same three ablation rows as TinyLlama, but on the paper-scale backbone (4-bit). Headline speedup target for Row 3 is 2.18x (paper §3.1 / Table 3).

In [13]:
# Table 3 ablation on Vicuna-7B (paper-scale).
# Saves results/table3_vicuna-7b-v1.5.json (consumed by visualize.py).
%run code/benchmark.py --mode table3 --model_id lmsys/vicuna-7b-v1.5 --max_new_tokens 128 --checkpoint results/medusa_heads_vicuna.pt

Using device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


=== Table 3 Ablation — lmsys/vicuna-7b-v1.5 ===
Loaded head checkpoint from results/medusa_heads_vicuna.pt

--- Greedy baseline (shared across ablation rows) ---
  [Write a Python script to sort ...] tps=52.99 time=2.42s tokens=128
  [Explain the theory of relativi...] tps=51.73 time=2.47s tokens=128
  [What are the benefits of eatin...] tps=51.80 time=2.36s tokens=122
  [Compose a short poem about the...] tps=51.97 time=2.46s tokens=128
  [Give me a step-by-step recipe ...] tps=51.36 time=2.49s tokens=128
Avg greedy TPS: 51.97

--- Medusa tree=none ---
  [Write a Python script to sort ...] acc=1.10 tps=90.5 | Here is a simple Python script to sort a list: ```python def sort_list(my_list):
  [Explain the theory of relativi...] acc=0.61 tps=71.3 | The theory of relativity is a scientific concept that explains how gravity works
  [What are the benefits of eatin...] acc=0.86 tps=84.9 | Eating a healthy diet offers numerous benefits for overall health and wellness, 
  [Compose a short poe

In [14]:
%run code/visualize.py

Loaded 2 benchmark(s), 2 Table 3 run(s), 4 comparison(s).
Saved /content/gdrive/MyDrive/CS4782/final_proj/results/head_accuracies.png
Saved /content/gdrive/MyDrive/CS4782/final_proj/results/speedup_comparison.png
Saved /content/gdrive/MyDrive/CS4782/final_proj/results/acceptance_histogram.png
Saved /content/gdrive/MyDrive/CS4782/final_proj/results/table3_comparison.png
Saved /content/gdrive/MyDrive/CS4782/final_proj/results/acceptance_comparison.png


<Figure size 640x480 with 0 Axes>